# Interactive RAG Q&A with Gradio

This notebook provides a **web interface** for querying both the **Text RAG** and **VLM RAG** pipelines interactively.

Instead of running command-line scripts, you get a browser-based UI with tabs for each pipeline:
- **Text RAG** — retrieves text chunks and answers with a text LLM
- **VLM RAG** — retrieves page images and answers with a Vision Language Model

Everything runs **locally** using Ollama (or optionally Azure).

In [ ]:
%pip install pymupdf sqlite-vec requests gradio --quiet

## Configuration

Set up the provider (local Ollama or Azure) and define helper functions for
embeddings, chat, and VLM chat.

In [ ]:
import os
import struct
import requests

# ---------- Provider selection ----------
PROVIDER = os.environ.get("PROVIDER", "local")

# ---------- Ollama (local) settings ----------
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_EMBED_MODEL = os.environ.get("OLLAMA_EMBED_MODEL", "phi3:mini")
OLLAMA_CHAT_MODEL = os.environ.get("OLLAMA_CHAT_MODEL", "phi3:mini")
OLLAMA_VLM_MODEL = os.environ.get("OLLAMA_VLM_MODEL", "qwen3.5:4b")

# ---------- Azure AI Foundry settings ----------
AZURE_API_KEY = os.environ.get("AZURE_API_KEY", "")
AZURE_CHAT_BASE_URL = os.environ.get("AZURE_CHAT_BASE_URL", "").rstrip("/")
AZURE_CHAT_DEPLOYMENT = os.environ.get("AZURE_CHAT_DEPLOYMENT", "gpt-4o")
AZURE_VLM_BASE_URL = os.environ.get("AZURE_VLM_BASE_URL", "").rstrip("/")
AZURE_VLM_DEPLOYMENT = os.environ.get("AZURE_VLM_DEPLOYMENT", "gpt-4o")
AZURE_EMBED_BASE_URL = os.environ.get("AZURE_EMBED_BASE_URL", "").rstrip("/")
AZURE_EMBED_DEPLOYMENT = os.environ.get("AZURE_EMBED_DEPLOYMENT", "text-embedding-ada-002")


# ---------- Helper functions ----------

def _azure_headers() -> dict:
    return {
        "Authorization": f"Bearer {AZURE_API_KEY}",
        "Content-Type": "application/json",
    }


def serialize_float32(vector: list[float]) -> bytes:
    """Serialize a list of floats into compact binary format for sqlite-vec."""
    return struct.pack(f"{len(vector)}f", *vector)


def get_embedding(text: str) -> list[float]:
    """Get embedding vector from the configured provider."""
    if PROVIDER == "azure":
        response = requests.post(
            f"{AZURE_EMBED_BASE_URL}/embeddings",
            headers=_azure_headers(),
            json={"model": AZURE_EMBED_DEPLOYMENT, "input": text},
        )
        response.raise_for_status()
        return response.json()["data"][0]["embedding"]
    else:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/embed",
            json={"model": OLLAMA_EMBED_MODEL, "input": text},
        )
        response.raise_for_status()
        return response.json()["embeddings"][0]


def chat(messages: list[dict], temperature: float = 0, max_tokens: int = 1024) -> str:
    """Send a text chat completion to the configured provider."""
    if PROVIDER == "azure":
        response = requests.post(
            f"{AZURE_CHAT_BASE_URL}/chat/completions",
            headers=_azure_headers(),
            json={
                "model": AZURE_CHAT_DEPLOYMENT,
                "messages": messages,
                "temperature": temperature,
                "max_completion_tokens": max_tokens,
            },
        )
    else:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/v1/chat/completions",
            json={
                "model": OLLAMA_CHAT_MODEL,
                "messages": messages,
                "temperature": temperature,
                "max_completion_tokens": max_tokens,
            },
        )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]


def vlm_chat(messages: list[dict], temperature: float = 0, max_tokens: int = 1024) -> str:
    """Send a chat completion to the VLM (vision model, supports image content parts)."""
    if PROVIDER == "azure":
        response = requests.post(
            f"{AZURE_VLM_BASE_URL}/chat/completions",
            headers=_azure_headers(),
            json={
                "model": AZURE_VLM_DEPLOYMENT,
                "messages": messages,
                "temperature": temperature,
                "max_completion_tokens": max_tokens,
            },
        )
    else:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/v1/chat/completions",
            json={
                "model": OLLAMA_VLM_MODEL,
                "messages": messages,
                "temperature": temperature,
                "max_completion_tokens": max_tokens,
            },
        )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]


print(f"Provider : {PROVIDER}")
if PROVIDER == "azure":
    print(f"Chat     : Azure ({AZURE_CHAT_DEPLOYMENT})")
    print(f"VLM      : Azure ({AZURE_VLM_DEPLOYMENT})")
    print(f"Embed    : Azure ({AZURE_EMBED_DEPLOYMENT})")
else:
    print(f"Chat     : Ollama ({OLLAMA_CHAT_MODEL})")
    print(f"VLM      : Ollama ({OLLAMA_VLM_MODEL})")
    print(f"Embed    : Ollama ({OLLAMA_EMBED_MODEL})")

## Text RAG Backend

This function embeds the user question, performs a vector search against the
text-chunk database (`rag/rag.db`), builds a prompt with the retrieved context,
and calls the text LLM.

In [ ]:
import sqlite3
import sqlite_vec
from pathlib import Path

TEXT_RAG_DB = "../rag/rag.db"


def text_rag_query(question: str) -> str:
    """Run the text RAG pipeline and return a formatted answer."""
    if not question.strip():
        return "Please enter a question."

    db_path = Path(TEXT_RAG_DB)
    if not db_path.exists():
        return (
            "ERROR: Text RAG database not found at rag/rag.db.\n"
            "Please run the ingest + embed steps from notebook 02 (or the rag/ scripts) first."
        )

    try:
        # 1. Embed the question
        query_vec = get_embedding(question)

        # 2. Vector search — top 3 chunks
        db = sqlite3.connect(str(db_path))
        db.enable_load_extension(True)
        sqlite_vec.load(db)
        db.enable_load_extension(False)

        rows = db.execute(
            """
            SELECT v.id, v.distance, c.text, c.source, c.page
            FROM vec_chunks v
            JOIN chunks c ON c.id = v.id
            WHERE v.embedding MATCH ?
                AND v.k = ?
            ORDER BY v.distance
            """,
            (serialize_float32(query_vec), 3),
        ).fetchall()
        db.close()

        results = [
            {"id": r[0], "distance": r[1], "text": r[2], "source": r[3], "page": r[4]}
            for r in rows
        ]

        if not results:
            return "No matching chunks found. The database may be empty."

        # 3. Build prompt
        context_text = "\n\n---\n\n".join(
            f"[From {c['source']}, page {c['page']}]\n{c['text']}"
            for c in results
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. Answer the user's question based on "
                    "the provided context ONLY, not internal knowledge. If the context "
                    "doesn't contain enough information, say so. "
                    "Cite which page the information comes from when possible."
                ),
            },
            {
                "role": "user",
                "content": f"Context:\n{context_text}\n\n---\n\nQuestion: {question}",
            },
        ]

        # 4. Call the LLM
        answer = chat(messages)

        # 5. Format the output
        output_parts = ["=== Retrieved Chunks ==="]
        for i, r in enumerate(results, 1):
            snippet = r["text"][:200].replace("\n", " ")
            output_parts.append(
                f"\n[{i}] (distance: {r['distance']:.4f}) "
                f"{r['source']}, page {r['page']}\n    {snippet}..."
            )
        output_parts.append("\n\n=== Answer ===")
        output_parts.append(answer)

        return "\n".join(output_parts)

    except Exception as e:
        return f"ERROR: {e}"


print("text_rag_query() ready.")

## VLM RAG Backend

This function embeds the user question, performs a vector search against the
page-image database (`vlm-rag/rag.db`), deduplicates by page, builds a
multi-modal prompt with page images, and calls the Vision LLM.

In [ ]:
VLM_RAG_DB = "../vlm-rag/rag.db"


def vlm_rag_query(question: str) -> str:
    """Run the VLM RAG pipeline and return a formatted answer."""
    if not question.strip():
        return "Please enter a question."

    db_path = Path(VLM_RAG_DB)
    if not db_path.exists():
        return (
            "ERROR: VLM RAG database not found at vlm-rag/rag.db.\n"
            "Please run the ingest + embed steps from notebook 03 (or the vlm-rag/ scripts) first."
        )

    try:
        # 1. Embed the question
        query_vec = get_embedding(question)

        # 2. Vector search — top 3 pages (fetch extra candidates for dedup)
        db = sqlite3.connect(str(db_path))
        db.enable_load_extension(True)
        sqlite_vec.load(db)
        db.enable_load_extension(False)

        rows = db.execute(
            """
            SELECT v.embed_id, v.distance, e.page_id, e.source_type,
                   p.text, p.description, p.image_b64, p.source, p.page
            FROM vec_pages v
            JOIN embed_index e ON e.embed_id = v.embed_id
            JOIN pages p ON p.id = e.page_id
            WHERE v.embedding MATCH ?
                AND v.k = ?
            ORDER BY v.distance
            """,
            (serialize_float32(query_vec), 6),
        ).fetchall()
        db.close()

        # Deduplicate by page — keep best match per page
        seen_pages = set()
        results = []
        for r in rows:
            page_id = r[2]
            if page_id in seen_pages:
                continue
            seen_pages.add(page_id)
            results.append({
                "id": page_id, "distance": r[1], "matched_on": r[3],
                "text": r[4], "description": r[5],
                "image_b64": r[6], "source": r[7], "page": r[8],
            })
            if len(results) >= 3:
                break

        if not results:
            return "No matching pages found. The database may be empty."

        # 3. Build multi-modal prompt with page images
        content_parts = []
        for r in results:
            content_parts.append({
                "type": "text",
                "text": f"[Page {r['page']} from {r['source']}]\nDescription: {r['description']}",
            })
            content_parts.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{r['image_b64']}",
                },
            })

        content_parts.append({
            "type": "text",
            "text": f"\n---\n\nQuestion: {question}",
        })

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. Answer the user's question based on "
                    "the provided page images ONLY, not internal knowledge. If the images "
                    "don't contain enough information, say so. "
                    "Cite which page the information comes from when possible."
                ),
            },
            {
                "role": "user",
                "content": content_parts,
            },
        ]

        # 4. Call the VLM
        answer = vlm_chat(messages)

        # 5. Format the output
        output_parts = ["=== Retrieved Pages ==="]
        for i, r in enumerate(results, 1):
            desc_snippet = r["description"][:200].replace("\n", " ")
            output_parts.append(
                f"\n[{i}] (distance: {r['distance']:.4f}, matched on: {r['matched_on']}) "
                f"{r['source']}, page {r['page']}\n    {desc_snippet}..."
            )
        output_parts.append("\n\n=== Answer ===")
        output_parts.append(answer)

        return "\n".join(output_parts)

    except Exception as e:
        return f"ERROR: {e}"


print("vlm_rag_query() ready.")

## Launch Gradio Interface

Run the cell below to start the Gradio web UI. A link will appear that you can
open in your browser. Use the tabs to switch between Text RAG and VLM RAG.

In [ ]:
import gradio as gr

with gr.Blocks(title="RAG Q&A Demo") as demo:
    gr.Markdown("# RAG Q&A Demo\nAsk questions about your ingested documents.")

    with gr.Tab("Text RAG"):
        gr.Markdown("Query the text-based RAG pipeline (uses `elc/session2/rag/rag.db`)")
        text_input = gr.Textbox(label="Your Question", placeholder="e.g., What is the attention mechanism?")
        text_output = gr.Textbox(label="Answer", lines=15)
        text_btn = gr.Button("Ask", variant="primary")
        text_btn.click(fn=text_rag_query, inputs=text_input, outputs=text_output)

    with gr.Tab("VLM RAG"):
        gr.Markdown("Query the vision RAG pipeline (uses `elc/session2/vlm-rag/rag.db`)")
        vlm_input = gr.Textbox(label="Your Question", placeholder="e.g., Summarize the key chart")
        vlm_output = gr.Textbox(label="Answer", lines=15)
        vlm_btn = gr.Button("Ask", variant="primary")
        vlm_btn.click(fn=vlm_rag_query, inputs=vlm_input, outputs=vlm_output)

demo.launch()

## Prerequisites

Before using this interface you must run the **ingest + embed** steps so that
the RAG databases exist:

- **Text RAG** (`rag/rag.db`) — run notebook **02** or the scripts in `elc/session2/rag/`
  (`01_ingest.py` then `02_embed.py`).
- **VLM RAG** (`vlm-rag/rag.db`) — run notebook **03** or the scripts in `elc/session2/vlm-rag/`
  (`01_ingest.py` then `02_embed.py`).

You also need **Ollama** running locally with the required models pulled:
```bash
ollama pull phi3:mini
ollama pull qwen3.5:4b
```